In [ ]:
#Comparisions of All Models

all_metrics = []

# 1. Evaluate Baselines
for name, model in Baseline_models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    all_metrics.append({
        "Group": "Baseline",
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, prob)
    })

# 2. Evaluate Candidates
for name, model in candidates.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    all_metrics.append({
        "Group": "Candidate",
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, prob)
    })

# 3. Evaluate Final SMOTE XGBoost Pipeline (using optimal threshold 0.39)
y_pred_final = (y_probs >= 0.39).astype(int)
all_metrics.append({
    "Group": "Final model",
    "Model": "SMOTE-XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred_final),
    "Precision": precision_score(y_test, y_pred_final),
    "Recall": recall_score(y_test, y_pred_final),
    "F1-Score": f1_score(y_test, y_pred_final),
    "ROC AUC": roc_auc_score(y_test, y_probs)
})

# Combine and Display
comparison_table = pd.DataFrame(all_metrics)
display(comparison_table.round(4).style.background_gradient(cmap='coolwarm', subset=["Accuracy", "Precision", "Recall", "F1-Score", "ROC AUC"]))

In [ ]:

plt.figure(figsize=(10, 8))

# 1. Plot Baseline Models
for name, model in Baseline_models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Baseline {name} (AUC = {roc_auc:.4f})')

# 2. Plot Candidate Models
for name, model in candidates.items():
    if name != "Logistic Regression":  # Skip duplicate LR
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])
        pipe.fit(X_train, y_train)
        probs = pipe.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Candidate {name} (AUC = {roc_auc:.4f})')

# 3. Plot Final SMOTE XGBoost Pipeline
fpr_final, tpr_final, _ = roc_curve(y_test, y_probs)
roc_auc_final = auc(fpr_final, tpr_final)
plt.plot(fpr_final, tpr_final, label=f' SMOTE-XGBoost (AUC = {roc_auc_final:.4f})', linewidth=3, linestyle='--')

# Plot Diagonal Random Guess Line
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('Comparison of ROC-AUC Curves Across All Models', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()